In [71]:
import xgboost as xgb
import pandas as pd
import os
from sklearn.model_selection import train_test_split

def fantasy_football_predictions(previous_year, model_filepath, input_filepath, output_filepath, position, columns):
    # Ensure output directory exists
    output_dir = os.path.dirname(output_filepath)
    os.makedirs(output_dir, exist_ok=True)

    # Predict on 2024 data for 2025
    model = xgb.Booster()
    model.load_model(model_filepath)
    players = pd.read_csv(input_filepath)
    #players = players[players['Year'] == previous_year]
    players_x = players[columns]
    dtest = xgb.DMatrix(players_x)
    probabilities = model.predict(dtest)
    players['great_season_probability_prediction'] = probabilities
    players = players[players['Pos'] == position]
    players = players[players['GS'] >= 6]

    # Save predictions as .csv file
    players.sort_values('great_season_probability_prediction', ascending=False).to_csv(output_filepath)

def fantasy_football_rb_predictions(previous_year, model_filepath, input_filepath_rush, input_filepath_rec, output_filepath, columns):
    # Ensure output directory exists
    output_dir = os.path.dirname(output_filepath)
    os.makedirs(output_dir, exist_ok=True)

    # Predict on 2024 data for 2025
    model = xgb.Booster()
    model.load_model(model_filepath)
    rush = pd.read_csv(input_filepath_rush)
    rec = pd.read_csv(input_filepath_rec)
    #rush = rush[rush['GS'] >= 8]
    #rec = rec[rec['GS'] >= 8]
    players = pd.merge(
        rush.rename(columns=lambda x: 'Rushing_' + x if x != 'Player' else x),
        rec.rename(columns=lambda x: 'Receiving_' + x if x != 'Player' else x),
        on='Player',
        how='inner'
    )
    players_x = players[columns]
    dtest = xgb.DMatrix(players_x)
    probabilities = model.predict(dtest)
    players['great_season_probability_prediction'] = probabilities
    
    # Save predictions as .csv file
    players.sort_values('great_season_probability_prediction', ascending=False).to_csv(output_filepath)

In [72]:
# QBs
previous_year = 2024
model_filepath = 'models/model_qbs.xgb'
input_filepath = 'data/adv_stats/pass_adv_stats_'+str(previous_year)+'.csv'
output_filepath = 'projections/'+str(previous_year+1)+'_xgboost_qb_rankings.csv'
columns = [
    'OnTgt%', 'Yds/Scr', 'CAY/PA'
    ]
position = 'QB'

fantasy_football_predictions(previous_year, model_filepath, input_filepath, output_filepath, position, columns)

In [73]:
# RBs
previous_year = 2024
model_filepath = 'models/model_rbs.xgb'
input_filepath_rush = 'data/adv_stats/rush_adv_'+str(previous_year)+'.csv'
input_filepath_rec = 'data/adv_stats/rec_adv_'+str(previous_year)+'.csv'
output_filepath = 'projections/'+str(previous_year+1)+'_xgboost_rb_rankings.csv'
columns = [
    'Rushing_YAC', 'Rushing_BrkTkl', 'Receiving_YBC',  'Receiving_BrkTkl', 'Receiving_Drop', 'Receiving_Int', 'Rushing_YAC/Att', 'Receiving_ADOT',
    'Rushing_YBC', 'Rushing_YBC/Att','Rushing_Att/Br',
    'Receiving_YAC', 'Receiving_YBC/R','Receiving_YAC/R','Receiving_Rec/Br','Receiving_Drop%',
    ]

fantasy_football_rb_predictions(previous_year, model_filepath, input_filepath_rush, input_filepath_rec, output_filepath, columns)

In [74]:
# WRs
previous_year = 2024
model_filepath = 'models/model_wrs.xgb'
input_filepath = 'data/adv_stats/rec_adv_'+str(previous_year)+'.csv'
output_filepath = 'projections/'+str(previous_year+1)+'_xgboost_wr_rankings.csv'
columns = [
    'YBC/R','YAC/R','ADOT','Rec/Br','Drop%'
    ]
position = 'WR'

fantasy_football_predictions(previous_year, model_filepath, input_filepath, output_filepath, position, columns)

In [75]:
# TEs
previous_year = 2024
model_filepath = 'models/model_tes.xgb'
input_filepath = 'data/adv_stats/rec_adv_'+str(previous_year)+'.csv'
output_filepath = 'projections/'+str(previous_year+1)+'_xgboost_te_rankings.csv'
columns = [
    'YBC/R','YAC/R','ADOT','Rec/Br','Drop%'
    ]
position = 'TE'

fantasy_football_predictions(previous_year, model_filepath, input_filepath, output_filepath, position, columns)